# Chapter 6: Tangent Space, Nonsingularity, and Dimension

Source orientation: Undergraduate Algebraic Geometry, printed pp. 101-108, PDF pages 99-106, sections 6.1-6.12 and exercises. The source span was used for topic order and terminology only; the explanations, examples, computations, and visuals below are original.

## Chapter Goal

The chapter turns a local geometric question into linear algebra: at a point `P`, keep only the first-order parts of the equations and read the tangent space as the common kernel of the resulting Jacobian rows. A point is nonsingular when this first-order approximation has the generic dimension of the variety. A point is singular when the first-order approximation is too large, which means it admits more infinitesimal directions than the nearby geometry actually has.

By the end of the notebook you should be able to compute tangent hyperplanes, find singular loci, read tangent-space dimension from Jacobian rank, distinguish tangent dimension from local dimension and tangent cone data, recognize why `m_P/m_P^2` makes the tangent space intrinsic, and use a blowup chart to inspect a strict transform.


In [ ]:
from pathlib import Path
import sys


def find_book_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (
            (candidate / "AGENTS.md").exists()
            and (candidate / "scripts" / "validate_uag_course.py").exists()
            and (candidate / "utils").exists()
        ):
            return candidate
    raise RuntimeError("Could not locate Undergraduate-Algebraic-Geometry course root")


BOOK_ROOT = find_book_root(Path.cwd())
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-06"
FIG_DIR = ARTIFACT_ROOT / "figures"
HTML_DIR = ARTIFACT_ROOT / "html"
CHECK_DIR = ARTIFACT_ROOT / "checks"
TABLE_DIR = ARTIFACT_ROOT / "tables"
for folder in [FIG_DIR, HTML_DIR, CHECK_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

artifact_paths = {"figures": [], "html": [], "checks": [], "tables": []}
chapter_metrics = {}

print(f"BOOK_ROOT = {BOOK_ROOT}")
print(f"ARTIFACT_ROOT = {ARTIFACT_ROOT.relative_to(BOOK_ROOT)}")


## Computational Translation Guide

| Book idea | Computational form | What to inspect |
| --- | --- | --- |
| First-order part of `f` at `P` | `grad(f)(P) dot (X - P)` | Which directions make the pulled-back one-variable polynomial have a double root at `T = 0`. |
| Tangent space to `V(f_1,...,f_m)` | kernel of the Jacobian matrix at `P` | Rank drop means extra infinitesimal directions. |
| Singular locus | common zero set of the equations and all relevant Jacobian minors | It is closed because minors are polynomial equations. |
| Dimension | generic/minimal tangent dimension on a dense open set | Singular points can have larger tangent spaces without increasing local dimension. |
| Intrinsic tangent space | dual of `m_P/m_P^2` | First-order functions do not depend on the chosen embedding. |
| Blowup chart `x = u, y = u v` | factor `f(u, u v) = u^m f_1(u, v)` | The exceptional line stores tangent directions; `f_1 = 0` is the strict transform in this chart. |

The notebook uses SymPy for exact derivatives, ranks, factorizations, and identities; Matplotlib for durable labeled diagrams; NetworkX for the proof-dependency scaffold; and Plotly for the interactive blowup laboratory.


In [ ]:
import csv
import json
import math

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sympy as sp

from utils.artifacts import display_artifact, image_stats
from utils.validation import validate_chapter_outputs

PALETTE = {
    "ink": "#23343b",
    "blue": "#2f6f9f",
    "teal": "#25897a",
    "gold": "#c9972b",
    "red": "#b84a62",
    "purple": "#6d5b88",
    "gray": "#6f777d",
}

x, y, z = sp.symbols("x y z")
X, Y, Z = sp.symbols("X Y Z")
u, v, T = sp.symbols("u v T")


def save_figure(fig, filename: str) -> Path:
    path = FIG_DIR / filename
    fig.savefig(path, dpi=165, bbox_inches="tight")
    plt.close(fig)
    artifact_paths["figures"].append(path)
    return path


def write_json(data, filename: str) -> Path:
    path = CHECK_DIR / filename
    path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    artifact_paths["checks"].append(path)
    return path


def write_csv(rows, filename: str) -> Path:
    path = TABLE_DIR / filename
    fieldnames = sorted({key for row in rows for key in row})
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    artifact_paths["tables"].append(path)
    return path


def tangent_linear_part(poly, variables, point):
    return sp.expand(sum(sp.diff(poly, var).subs(point) * (var - point[var]) for var in variables))


def jacobian_at(polys, variables, point):
    return sp.Matrix([[sp.diff(poly, var).subs(point) for var in variables] for poly in polys])


def tangent_dimension(polys, variables, point):
    return len(variables) - jacobian_at(polys, variables, point).rank()


def min_total_degree_part(poly, variables):
    expanded = sp.Poly(sp.expand(poly), *variables)
    terms = expanded.terms()
    min_degree = min(sum(exps) for exps, coeff in terms if coeff != 0)
    part = sum(
        coeff * math.prod(var ** exp for var, exp in zip(variables, exps, strict=True))
        for exps, coeff in terms
        if sum(exps) == min_degree
    )
    return sp.factor(part), int(min_degree)


def min_u_factor(poly_uv):
    poly = sp.Poly(sp.expand(poly_uv), u, v)
    min_power = min(exps[0] for exps, coeff in poly.terms() if coeff != 0)
    strict = sp.factor(sp.expand(poly_uv / (u ** min_power)))
    return int(min_power), strict


def grid_contour(ax, expr, xlim, ylim, *, color, linewidth=2.2, alpha=1.0):
    xs = np.linspace(xlim[0], xlim[1], 480)
    ys = np.linspace(ylim[0], ylim[1], 420)
    xx, yy = np.meshgrid(xs, ys)
    f_np = sp.lambdify((x, y), expr, "numpy")
    zz = f_np(xx, yy)
    ax.contour(xx, yy, zz, levels=[0], colors=[color], linewidths=linewidth, alpha=alpha)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal", adjustable="box")
    ax.axhline(0, color="#c8c8c8", lw=0.8, zorder=0)
    ax.axvline(0, color="#c8c8c8", lw=0.8, zorder=0)


## Tangent Hyperplanes Are First-Order Tests

For a hypersurface `V(f)` and a point `P`, a line through `P` with direction `b` has the form `P + b T`. The point `P` is a double root of the restricted polynomial `f(P + b T)` exactly when the derivative at `T = 0` vanishes. Algebraically this derivative is the dot product `grad(f)(P) dot b`, so the allowed directions form the tangent hyperplane.

The figure uses the parabola `f = y - x^2` because its tangent lines are easy to verify exactly. Inspect how every drawn tangent line is controlled by the same first-order recipe, not by a limiting analytic argument.


In [ ]:
f_parabola = y - x**2
sample_points = [
    {x: sp.Rational(-3, 2), y: sp.Rational(9, 4)},
    {x: sp.Rational(-1, 2), y: sp.Rational(1, 4)},
    {x: sp.Rational(1, 2), y: sp.Rational(1, 4)},
    {x: sp.Rational(3, 2), y: sp.Rational(9, 4)},
]

xs = np.linspace(-2.2, 2.2, 400)
fig, ax = plt.subplots(figsize=(8.4, 5.8))
ax.plot(xs, xs**2, color=PALETTE["ink"], lw=2.8, label="V(y - x^2)")
colors = [PALETTE["blue"], PALETTE["teal"], PALETTE["gold"], PALETTE["red"]]

for idx, point in enumerate(sample_points):
    a = float(point[x])
    b = float(point[y])
    tangent = tangent_linear_part(f_parabola, [x, y], point)
    tangent_y = sp.solve(sp.Eq(tangent, 0), y)[0]
    yy = sp.lambdify(x, tangent_y, "numpy")(xs)
    ax.plot(xs, yy, lw=1.7, color=colors[idx], alpha=0.9)
    ax.scatter([a], [b], s=48, color="white", edgecolor=PALETTE["ink"], zorder=5)
    ax.text(a + 0.05, b + 0.18, f"P{idx + 1}", fontsize=9)

P = {x: sp.Integer(1), y: sp.Integer(1)}
g_tangent = sp.expand(f_parabola.subs({x: P[x] + T, y: P[y] + 2 * T}))
g_transverse = sp.expand(f_parabola.subs({x: P[x] + T, y: P[y]}))
assert g_tangent.subs(T, 0) == 0 and sp.diff(g_tangent, T).subs(T, 0) == 0
assert g_transverse.subs(T, 0) == 0 and sp.diff(g_transverse, T).subs(T, 0) != 0

ax.set_title("Tangent directions detected by first-order vanishing")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_ylim(-1.0, 4.4)
ax.grid(True, alpha=0.18)
ax.legend(loc="upper center", frameon=False)
ax.text(
    -2.15,
    3.7,
    "At P=(1,1):\nline direction (1,2) gives f(P+Tb)=O(T^2)\nline direction (1,0) gives a simple root",
    fontsize=9,
    va="top",
    bbox={"boxstyle": "round,pad=0.35", "fc": "white", "ec": "#dddddd"},
)

path = save_figure(fig, "tangent-hyperplane-sweep.png")
chapter_metrics["tangent_line_test"] = {
    "point": "(1, 1)",
    "tangent_restriction": str(g_tangent),
    "transverse_restriction": str(g_transverse),
    "tangent_linear_part": str(tangent_linear_part(f_parabola, [x, y], P)),
}
display_artifact(path, width=760)


## Singular Loci and Tangent Cones

For a plane hypersurface, a singular point is a common zero of `f`, `f_x`, and `f_y`. The tangent space at such a point may be all of `A^2`, so the first nonzero homogeneous part of `f` gives a sharper local picture: the tangent cone.

The gallery compares a smooth point, a cusp, a node, and a tacnode-like reducible example. Inspect the red tangent-cone lines. A cusp and tacnode both have a doubled tangent line, while the node has two distinct tangent directions.


In [ ]:
curve_examples = [
    {"name": "smooth branch", "poly": y - x**2, "xlim": (-1.8, 1.8), "ylim": (-0.6, 2.4), "cone_lines": [(0.0, 0.0)], "note": "one tangent line"},
    {"name": "cusp", "poly": y**2 - x**3, "xlim": (-0.8, 2.0), "ylim": (-1.4, 1.4), "cone_lines": [(0.0, 0.0)], "note": "tangent space too large"},
    {"name": "node", "poly": y**2 - x**2 - x**3, "xlim": (-1.2, 1.8), "ylim": (-1.5, 1.5), "cone_lines": [(1.0, 0.0), (-1.0, 0.0)], "note": "two tangent directions"},
    {"name": "tacnode toy", "poly": y**2 - x**4, "xlim": (-1.6, 1.6), "ylim": (-1.3, 1.3), "cone_lines": [(0.0, 0.0)], "note": "two branches share tangent"},
]

fig, axes = plt.subplots(2, 2, figsize=(10.4, 8.0))
singular_rows = []
for ax, example in zip(axes.ravel(), curve_examples, strict=True):
    poly = example["poly"]
    grid_contour(ax, poly, example["xlim"], example["ylim"], color=PALETTE["ink"])
    cone, multiplicity = min_total_degree_part(poly, [x, y])
    singulars = sp.solve([poly, sp.diff(poly, x), sp.diff(poly, y)], [x, y], dict=True)
    singulars = [sol for sol in singulars if all(value.is_real is not False for value in sol.values())]
    if singulars:
        for sol in singulars:
            ax.scatter([float(sol[x])], [float(sol[y])], s=58, color=PALETTE["red"], zorder=6)
    else:
        ax.scatter([0], [0], s=46, color="white", edgecolor=PALETTE["teal"], zorder=6)

    line_x = np.linspace(example["xlim"][0], example["xlim"][1], 2)
    for slope, intercept in example["cone_lines"]:
        ax.plot(line_x, slope * line_x + intercept, color=PALETTE["red"], lw=1.4, ls="--", alpha=0.88)
    ax.set_title(f"{example['name']}: {sp.factor(poly)}")
    ax.text(
        0.03,
        0.95,
        f"lowest part: {cone}\n{example['note']}",
        transform=ax.transAxes,
        va="top",
        fontsize=8.7,
        bbox={"boxstyle": "round,pad=0.3", "fc": "white", "ec": "#dddddd", "alpha": 0.94},
    )
    singular_rows.append(
        {
            "curve": example["name"],
            "polynomial": str(sp.factor(poly)),
            "singular_points": str([{str(k): str(vv) for k, vv in sol.items()} for sol in singulars]),
            "tangent_cone_at_origin": str(cone),
            "multiplicity_at_origin": multiplicity,
        }
    )

fig.suptitle("Singular loci and tangent cones for plane curves", y=0.995, fontsize=14)
fig.tight_layout()
path = save_figure(fig, "singular-curve-gallery.png")
chapter_metrics["singular_curve_gallery"] = singular_rows
assert any(row["curve"] == "cusp" and row["multiplicity_at_origin"] == 2 for row in singular_rows)
display_artifact(path, width=820)


## Jacobian Rank, Dimension, and Where Jumps Happen

For equations `f_1,...,f_m` in `n` variables, the tangent space is the kernel of the `m x n` Jacobian at the point, so `dim T_P V = n - rank J(P)`.

The left panel is the irreducible cusp parametrized by `(t^2, t^3)`: local dimension is one everywhere, but tangent dimension jumps from one to two at the cusp. The right panel is a reducible rank-strata toy `V(xz, yz)` in `A^3`; it is included as a warning that component dimension and tangent dimension are different diagnostics.


In [ ]:
f_cusp = y**2 - x**3
cusp_points = [{x: sp.Rational(i, 4) ** 2, y: sp.Rational(i, 4) ** 3} for i in range(-6, 7)]
cusp_dims = [tangent_dimension([f_cusp], [x, y], point) for point in cusp_points]
assert min(cusp_dims) == 1 and max(cusp_dims) == 2

polys_rank_toy = [x * z, y * z]
rank_samples = []
for a in [-1, 0, 1]:
    for b in [-1, 0, 1]:
        point = {x: sp.Integer(a), y: sp.Integer(b), z: sp.Integer(0)}
        rank_samples.append((a, b, 0, tangent_dimension(polys_rank_toy, [x, y, z], point)))
for c in [-1, -0.5, 0.5, 1]:
    point = {x: 0, y: 0, z: sp.Rational(str(c))}
    rank_samples.append((0, 0, float(c), tangent_dimension(polys_rank_toy, [x, y, z], point)))
origin_dim = tangent_dimension(polys_rank_toy, [x, y, z], {x: 0, y: 0, z: 0})
assert origin_dim == 3

fig = plt.figure(figsize=(12.0, 5.8))
ax1 = fig.add_subplot(1, 2, 1)
parameter = np.linspace(-1.6, 1.6, 500)
ax1.plot(parameter**2, parameter**3, color=PALETTE["ink"], lw=2.4)
for point, dim in zip(cusp_points, cusp_dims, strict=True):
    color = PALETTE["red"] if dim == 2 else PALETTE["blue"]
    ax1.scatter([float(point[x])], [float(point[y])], s=54 if dim == 2 else 28, color=color, zorder=5)
ax1.set_title("Cusp: tangent dimension jumps at one point")
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.set_xlim(-0.12, 2.75)
ax1.set_ylim(-4.35, 4.35)
ax1.grid(True, alpha=0.18)
ax1.text(
    0.04,
    0.94,
    "generic dim T_P = 1\nat origin dim T_P = 2\nlocal curve dimension remains 1",
    transform=ax1.transAxes,
    va="top",
    fontsize=9,
    bbox={"boxstyle": "round,pad=0.32", "fc": "white", "ec": "#dddddd"},
)

ax2 = fig.add_subplot(1, 2, 2, projection="3d")
for dim in [1, 2, 3]:
    pts = np.array([(a, b, c) for a, b, c, d in rank_samples + [(0, 0, 0, origin_dim)] if d == dim], dtype=float)
    if len(pts):
        ax2.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=55 if dim == 3 else 34, label=f"dim T = {dim}")
plane_x, plane_y = np.meshgrid(np.linspace(-1, 1, 2), np.linspace(-1, 1, 2))
ax2.plot_surface(plane_x, plane_y, np.zeros_like(plane_x), alpha=0.13, color=PALETTE["teal"], edgecolor="none")
ax2.plot([0, 0], [0, 0], [-1.15, 1.15], color=PALETTE["purple"], lw=2.0, alpha=0.75)
ax2.set_title("Rank strata for V(xz, yz)")
ax2.set_xlabel("x")
ax2.set_ylabel("y")
ax2.set_zlabel("z")
ax2.legend(loc="upper left", fontsize=8)
ax2.view_init(elev=21, azim=-58)

fig.suptitle("Jacobian rank controls tangent-space dimension", y=1.02, fontsize=14)
fig.tight_layout()
path = save_figure(fig, "jacobian-rank-strata.png")
chapter_metrics["jacobian_rank_strata"] = {
    "cusp_tangent_dimensions": sorted(set(cusp_dims)),
    "rank_toy_origin_tangent_dimension": origin_dim,
    "rank_toy_polynomials": [str(poly) for poly in polys_rank_toy],
}
display_artifact(path, width=860)


## Intrinsic Tangent Space: Why `m_P/m_P^2` Appears

The ambient Jacobian computation is convenient, but the tangent space is not supposed to depend on the embedding or the chosen affine chart. The intrinsic replacement is the vector space of first-order functions at `P`: the maximal ideal `m_P` modulo functions that vanish to second order, `m_P^2`. Its dual is the tangent space.

The proof-flow diagram records the algebraic mechanism. The key point is that first-order parts survive in `m_P/m_P^2`, while quadratic and higher terms disappear. For a standard open containing `P`, localizing by a function nonzero at `P` does not change this first-order quotient.


In [ ]:
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(12.2, 4.8))
ax.set_xlim(0, 12)
ax.set_ylim(0, 4.8)
ax.axis("off")

boxes = {
    "ambient": (0.25, 3.0, 1.8, 1.05, "ambient maximal\nideal M_P", PALETTE["blue"]),
    "linear": (2.55, 3.0, 1.85, 1.05, "linear part map\nd: M_P -> (k^n)^*", PALETTE["teal"]),
    "kernel": (2.55, 0.75, 1.85, 1.05, "kernel\nM_P^2", PALETTE["gray"]),
    "relations": (4.95, 3.0, 1.7, 1.05, "add equations\nI(V)", PALETTE["gold"]),
    "quotient": (7.05, 3.0, 1.8, 1.05, "first-order\nfunctions\nm_P/m_P^2", PALETTE["purple"]),
    "dual": (9.35, 3.0, 1.75, 1.05, "dual tangent\nspace T_P V", PALETTE["red"]),
    "open": (7.05, 0.75, 1.8, 1.05, "standard open\nf(P) != 0", "#8a8f3a"),
}

for key, (x0, y0, w, h, label, color) in boxes.items():
    patch = FancyBboxPatch((x0, y0), w, h, boxstyle="round,pad=0.16,rounding_size=0.08", fc=color, ec="none", alpha=0.95)
    ax.add_patch(patch)
    ax.text(x0 + w / 2, y0 + h / 2, label, ha="center", va="center", color="white", fontsize=9)


def center(key, side="right"):
    x0, y0, w, h, *_ = boxes[key]
    if side == "right":
        return x0 + w, y0 + h / 2
    if side == "left":
        return x0, y0 + h / 2
    if side == "top":
        return x0 + w / 2, y0 + h
    if side == "bottom":
        return x0 + w / 2, y0
    return x0 + w / 2, y0 + h / 2


def arrow(a, b, label, aside="right", bside="left", dy=-0.68):
    ax.annotate("", xy=center(b, bside), xytext=center(a, aside), arrowprops={"arrowstyle": "-|>", "lw": 1.8, "color": "#333333"})
    x1, y1 = center(a, aside)
    x2, y2 = center(b, bside)
    ax.text((x1 + x2) / 2, (y1 + y2) / 2 + dy, label, ha="center", va="center", fontsize=8.4, color="#222222")

arrow("ambient", "linear", "keep degree 1")
arrow("linear", "relations", "relations from equations")
arrow("relations", "quotient", "quotient")
arrow("quotient", "dual", "dualize")
arrow("linear", "kernel", "kills degree >= 2", aside="bottom", bside="top", dy=0.0)
arrow("open", "quotient", "same first-order data", aside="top", bside="bottom", dy=0.0)

ax.set_title("Intrinsic tangent space as first-order functions modulo second-order functions", fontsize=13, pad=18)
path = save_figure(fig, "intrinsic-tangent-quotient-flow.png")

origin_quotient_dim = tangent_dimension([f_cusp], [x, y], {x: 0, y: 0})
smooth_quotient_dim = tangent_dimension([f_cusp], [x, y], {x: 1, y: 1})
assert origin_quotient_dim == 2
assert smooth_quotient_dim == 1
chapter_metrics["intrinsic_quotient_dimensions"] = {
    "cusp_origin_dim_m_mod_m2": origin_quotient_dim,
    "cusp_smooth_dim_m_mod_m2": smooth_quotient_dim,
}
display_artifact(path, width=850)


## Projective Tangent Hyperplanes and Euler's Identity

For a homogeneous polynomial `F` of degree `d`, Euler's identity `sum X_i F_{X_i} = dF` explains why the projective tangent hyperplane at `P` can be written as `sum F_{X_i}(P) X_i = 0`. On the affine chart `Z = 1`, this agrees with the usual affine tangent line.

The figure uses the projective cubic `X^3 + Y^3 - Z^3 = 0`. At `P = (1:0:1)`, the projective tangent is `X - Z = 0`, which becomes the vertical affine line `x = 1` in the chart `Z = 1`.


In [ ]:
F_projective = X**3 + Y**3 - Z**3
degree = 3
euler_residual = sp.expand(X * sp.diff(F_projective, X) + Y * sp.diff(F_projective, Y) + Z * sp.diff(F_projective, Z) - degree * F_projective)
assert euler_residual == 0
P_projective = {X: 1, Y: 0, Z: 1}
assert F_projective.subs(P_projective) == 0
tangent_projective = sp.expand(sum(sp.diff(F_projective, var).subs(P_projective) * var for var in [X, Y, Z]))
assert tangent_projective == 3 * X - 3 * Z

f_affine_fermat = x**3 + y**3 - 1
affine_tangent = tangent_linear_part(f_affine_fermat, [x, y], {x: 1, y: 0})
assert affine_tangent == 3 * x - 3

fig, ax = plt.subplots(figsize=(7.4, 5.6))
grid_contour(ax, f_affine_fermat, (-1.6, 1.6), (-1.6, 1.6), color=PALETTE["ink"])
ax.axvline(1, color=PALETTE["red"], lw=2.0, ls="--", label="tangent x=1")
ax.scatter([1], [0], s=58, color=PALETTE["red"], zorder=5)
ax.text(1.05, 0.08, "P=(1,0)", fontsize=9)
ax.set_title("Projective tangent checked in the affine chart Z=1")
ax.set_xlabel("x = X/Z")
ax.set_ylabel("y = Y/Z")
ax.legend(frameon=False)
ax.text(
    -1.53,
    -1.35,
    "F=X^3+Y^3-Z^3\nEuler residual = 0\nprojective tangent: X-Z=0",
    fontsize=9,
    bbox={"boxstyle": "round,pad=0.34", "fc": "white", "ec": "#dddddd"},
)
path = save_figure(fig, "projective-euler-tangent-chart.png")
chapter_metrics["projective_euler_identity"] = {
    "euler_residual": str(euler_residual),
    "projective_tangent_at_(1:0:1)": str(tangent_projective),
    "affine_chart_tangent": str(affine_tangent),
}
display_artifact(path, width=700)


## Blowup Charts and Strict Transforms

In the chart `x = u`, `y = u v`, the exceptional line is `u = 0`. If `f` has multiplicity `m` at the origin, then `f(u, u v)` is divisible by `u^m`. Removing that exceptional factor gives the strict transform.

The three panels show what the chapter's blowup examples are meant to reveal: a smooth branch meets the exceptional line at the point representing its tangent direction; a node separates into two points over the origin; and a cusp becomes nonsingular but remains tangent to the exceptional line in this chart. This chart records nonvertical tangent directions; a second chart handles directions with `x = 0`.


In [ ]:
blowup_examples = [
    {"name": "smooth branch", "poly": y - sp.Rational(1, 2) * x, "color": PALETTE["teal"], "intersections": [(0, sp.Rational(1, 2))]},
    {"name": "node", "poly": y**2 - x**2 - x**3, "color": PALETTE["blue"], "intersections": [(0, -1), (0, 1)]},
    {"name": "cusp", "poly": y**2 - x**3, "color": PALETTE["red"], "intersections": [(0, 0)]},
]

blowup_rows = []
fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.8), sharex=True, sharey=True)
for ax, example in zip(axes, blowup_examples, strict=True):
    transformed = sp.factor(example["poly"].subs({x: u, y: u * v}))
    exceptional_power, strict_transform = min_u_factor(transformed)
    original_cone, original_multiplicity = min_total_degree_part(example["poly"], [x, y])
    assert exceptional_power == original_multiplicity

    uu = np.linspace(-1.25, 1.45, 420)
    vv = np.linspace(-1.8, 1.8, 420)
    U, V = np.meshgrid(uu, vv)
    strict_np = sp.lambdify((u, v), strict_transform, "numpy")
    ax.contour(U, V, strict_np(U, V), levels=[0], colors=[example["color"]], linewidths=2.4)
    ax.axvline(0, color=PALETTE["ink"], lw=1.6, ls="--", label="exceptional u=0")
    for pu, pv in example["intersections"]:
        grad = [sp.diff(strict_transform, var).subs({u: pu, v: pv}) for var in [u, v]]
        assert any(component != 0 for component in grad)
        ax.scatter([float(pu)], [float(pv)], s=58, color=example["color"], edgecolor="white", zorder=6)
        ax.text(float(pu) + 0.05, float(pv) + 0.08, f"(0,{pv})", fontsize=8)
    ax.set_title(example["name"])
    ax.set_xlabel("u")
    ax.set_xlim(-1.2, 1.35)
    ax.set_ylim(-1.65, 1.65)
    ax.grid(True, alpha=0.16)
    ax.text(
        0.04,
        0.96,
        f"f(u,uv)={transformed}\nstrict: {strict_transform}",
        transform=ax.transAxes,
        va="top",
        fontsize=8.0,
        bbox={"boxstyle": "round,pad=0.25", "fc": "white", "ec": "#dddddd", "alpha": 0.95},
    )
    blowup_rows.append(
        {
            "example": example["name"],
            "original_polynomial": str(sp.factor(example["poly"])),
            "multiplicity_at_origin": exceptional_power,
            "transformed_polynomial": str(transformed),
            "strict_transform": str(strict_transform),
            "exceptional_intersections": str(example["intersections"]),
        }
    )
axes[0].set_ylabel("v = y/x")
fig.suptitle("Blowup chart x=u, y=uv: exceptional directions and strict transforms", y=1.03, fontsize=13)
fig.tight_layout()
path = save_figure(fig, "blowup-strict-transform.png")
chapter_metrics["blowup_strict_transforms"] = blowup_rows
display_artifact(path, width=920)


## Interactive Lab: Compare the Original Curve and Its Blowup Chart

Use the Plotly controls to switch among the smooth branch, node, and cusp. The left panel is the original curve near the origin; the right panel is the strict transform in the `x`-chart of the blowup. The fixed dashed line on the right is the exceptional line `u = 0`.

The lab is not a proof by itself. It is a way to inspect the algebraic factorization from the previous cell: where `f(u, u v)` loses a power of `u`, the remaining equation remembers the tangent direction data.


In [ ]:
def original_curve_samples(name):
    s = np.linspace(-1.25, 1.25, 360)
    if name == "smooth branch":
        return [(s, 0.5 * s, "smooth branch")]
    if name == "cusp":
        return [(s**2, s**3, "cusp parameter t")]
    if name == "node":
        xvals = np.linspace(0, 1.25, 260)
        yvals = xvals * np.sqrt(1 + xvals)
        return [(xvals, yvals, "upper branch"), (xvals, -yvals, "lower branch")]
    raise KeyError(name)


def strict_transform_samples(example):
    name = example["name"]
    if name == "smooth branch":
        s = np.linspace(-1.55, 1.55, 420)
        return [(s, np.full_like(s, 0.5), "strict transform")]
    if name == "node":
        vvals = np.linspace(-1.55, 1.55, 420)
        return [(vvals**2 - 1, vvals, "strict transform")]
    if name == "cusp":
        vvals = np.linspace(-1.25, 1.25, 360)
        return [(vvals**2, vvals, "strict transform")]
    raise KeyError(name)


fig = make_subplots(rows=1, cols=2, subplot_titles=("original (x,y)", "blowup chart (u,v)"))
visibility_blocks = []
for idx, example in enumerate(blowup_examples):
    visible = idx == 0
    block_indices = []
    for xs_plot, ys_plot, label in original_curve_samples(example["name"]):
        fig.add_trace(go.Scatter(x=xs_plot, y=ys_plot, mode="lines", name=f"{example['name']}: {label}", line={"color": example["color"], "width": 3}, visible=visible), row=1, col=1)
        block_indices.append(len(fig.data) - 1)
    for us_plot, vs_plot, label in strict_transform_samples(example):
        fig.add_trace(go.Scatter(x=us_plot, y=vs_plot, mode="lines", name=f"{example['name']}: {label}", line={"color": example["color"], "width": 3}, visible=visible), row=1, col=2)
        block_indices.append(len(fig.data) - 1)
    visibility_blocks.append(block_indices)

fig.add_trace(go.Scatter(x=[0, 0], y=[-1.7, 1.7], mode="lines", name="exceptional line u=0", line={"color": "#23343b", "dash": "dash", "width": 2}), row=1, col=2)
exceptional_index = len(fig.data) - 1

buttons = []
for idx, example in enumerate(blowup_examples):
    visible = [False] * len(fig.data)
    for trace_index in visibility_blocks[idx]:
        visible[trace_index] = True
    visible[exceptional_index] = True
    buttons.append({"label": example["name"], "method": "update", "args": [{"visible": visible}, {"title": f"Blowup laboratory: {example['name']}"}]})

fig.update_layout(
    title="Blowup laboratory: smooth branch",
    template="plotly_white",
    width=940,
    height=520,
    updatemenus=[{"buttons": buttons, "direction": "right", "x": 0.5, "xanchor": "center", "y": 1.15, "yanchor": "top"}],
    legend={"orientation": "h", "y": -0.18},
)
fig.update_xaxes(range=[-1.35, 1.45], zeroline=True)
fig.update_yaxes(range=[-1.7, 1.7], zeroline=True, scaleanchor="x", scaleratio=1, row=1, col=1)
fig.update_yaxes(range=[-1.7, 1.7], zeroline=True, scaleanchor="x2", scaleratio=1, row=1, col=2)
html_path = HTML_DIR / "blowup-strict-transform-lab.html"
fig.write_html(str(html_path), include_plotlyjs="cdn", full_html=True)
artifact_paths["html"].append(html_path)
chapter_metrics["interactive_lab"] = {"html": html_path.relative_to(BOOK_ROOT).as_posix(), "examples": [item["name"] for item in blowup_examples]}
display_artifact(html_path, height=560)


## Applied Lab Table: Run the Same Tests on Several Curves

The table below collects the notebook's local diagnostics in one place. It is deliberately small: each row is a curve or algebraic set where the exact Jacobian computation says something geometric that can be checked against a visual artifact.


In [ ]:
diagnostic_rows = []
for row in singular_rows:
    diagnostic_rows.append(
        {
            "object": row["curve"],
            "equations": row["polynomial"],
            "diagnostic": "singular locus and tangent cone",
            "result": f"singulars={row['singular_points']}; cone={row['tangent_cone_at_origin']}",
        }
    )
for row in blowup_rows:
    diagnostic_rows.append(
        {
            "object": row["example"],
            "equations": row["original_polynomial"],
            "diagnostic": "blowup x-chart strict transform",
            "result": f"u-power={row['multiplicity_at_origin']}; strict={row['strict_transform']}",
        }
    )

table_path = write_csv(diagnostic_rows, "tangent-space-examples.csv")
concept_rows = [
    {"concept": "gradient tangent hyperplanes", "artifact": "tangent-hyperplane-sweep.png", "check": "line restriction has double root iff direction is tangent"},
    {"concept": "singular locus", "artifact": "singular-curve-gallery.png", "check": "solve f, f_x, f_y"},
    {"concept": "Jacobian rank and tangent dimension", "artifact": "jacobian-rank-strata.png", "check": "dim T_P = n - rank J(P)"},
    {"concept": "intrinsic tangent space", "artifact": "intrinsic-tangent-quotient-flow.png", "check": "dim m_P/m_P^2 matches tangent dimension"},
    {"concept": "projective tangent space", "artifact": "projective-euler-tangent-chart.png", "check": "Euler identity for homogeneous F"},
    {"concept": "blowup charts", "artifact": "blowup-strict-transform.png", "check": "strict transforms nonsingular at exceptional intersections"},
]
concept_path = write_csv(concept_rows, "concepts.csv")
chapter_metrics["diagnostic_table_rows"] = len(diagnostic_rows)
display_artifact(table_path)
display_artifact(concept_path)


## Final Sanity Checks

The final cell checks core algebraic identities, artifact existence, path locality, nonzero size, nonblank image statistics, and concept-specific validation data for Jacobian ranks, tangent dimensions, Euler's identity, and strict-transform nonsingularity.


In [ ]:
noncheck_artifacts = [path for key, group in artifact_paths.items() if key != "checks" for path in group]
planned_check_paths = [CHECK_DIR / "artifact-summary.json", CHECK_DIR / "final-sanity.json"]
all_artifacts = [*noncheck_artifacts, *planned_check_paths]
assert all_artifacts, "no artifacts were generated"

image_summaries = [image_stats(path) for path in artifact_paths["figures"]]
for item in image_summaries:
    assert item["pixel_std"] > 2.0, f"image appears blank: {item['path']}"

assert sp.diff(g_tangent, T).subs(T, 0) == 0
assert sp.diff(g_transverse, T).subs(T, 0) != 0
assert origin_quotient_dim == tangent_dimension([f_cusp], [x, y], {x: 0, y: 0}) == 2
assert smooth_quotient_dim == tangent_dimension([f_cusp], [x, y], {x: 1, y: 1}) == 1
assert euler_residual == 0
assert tangent_projective.subs(P_projective) == 0
assert chapter_metrics["jacobian_rank_strata"]["rank_toy_origin_tangent_dimension"] == 3
for row in blowup_rows:
    assert int(row["multiplicity_at_origin"]) >= 1

artifact_summary = {
    "figures": [path.relative_to(BOOK_ROOT).as_posix() for path in artifact_paths["figures"]],
    "html": [path.relative_to(BOOK_ROOT).as_posix() for path in artifact_paths["html"]],
    "tables": [path.relative_to(BOOK_ROOT).as_posix() for path in artifact_paths["tables"]],
    "checks": [path.relative_to(BOOK_ROOT).as_posix() for path in planned_check_paths],
    "image_stats": image_summaries,
    "metrics": chapter_metrics,
}
summary_path = write_json(artifact_summary, "artifact-summary.json")
validation_summary = validate_chapter_outputs({
    "figures": artifact_paths["figures"],
    "html": artifact_paths["html"],
    "checks": [summary_path],
}, min_pngs=6)

final_sanity = {
    "artifact_summary": summary_path.relative_to(BOOK_ROOT).as_posix(),
    "validation_summary": validation_summary,
    "artifact_counts": {
        "figures": len(artifact_paths["figures"]),
        "html": len(artifact_paths["html"]),
        "tables": len(artifact_paths["tables"]),
        "checks": len(planned_check_paths),
    },
    "core_checks": {
        "tangent_line_double_root": True,
        "singular_cusp_tangent_dimension_jump": True,
        "intrinsic_m_mod_m2_dimensions": chapter_metrics["intrinsic_quotient_dimensions"],
        "projective_euler_identity": chapter_metrics["projective_euler_identity"],
        "strict_transforms_checked": [row["example"] for row in blowup_rows],
    },
}
final_path = write_json(final_sanity, "final-sanity.json")

for path in all_artifacts:
    assert path.exists(), f"missing artifact: {path}"
    assert path.stat().st_size > 256, f"artifact too small: {path}"
    assert path.resolve().is_relative_to(ARTIFACT_ROOT.resolve()), f"artifact escaped chapter root: {path}"

final_sanity


## Takeaways

- A tangent space is a first-order algebraic object: for hypersurfaces it is read from a gradient, and for several equations it is the kernel of a Jacobian.
- Singular points are exactly where the first-order approximation is too large relative to the generic dimension.
- Tangent cones refine the story at singular points because they remember the first nonzero homogeneous part after all linear terms vanish.
- The quotient `m_P/m_P^2` explains why the tangent space is intrinsic and why standard affine opens preserve it.
- Projective tangent hyperplanes are compatible with affine chart tangents because homogeneous polynomials satisfy Euler's identity.
- Blowups replace a point by tangent directions; strict transforms show whether branches separate or remain tangent to the exceptional line.
